In [ ]:
# ! pip install -q openai datasets pandas tqdm dotenv

### Imports

In [1]:
from datasets import load_dataset
from openai import OpenAI
import os
import json
import mlflow
import pandas as pd
from utils import generate_urls, calculate_invoice_accuracies, key_level_metrics
from prompt import register_prompt
from model import log_invoice_extraction_model

from dotenv import load_dotenv
from mlflow.models import make_metric


load_dotenv()

d:\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


True

### Config

In [2]:
MLFLOW_TRACKING_URI = "http://localhost:8080/"
MODEL_NAME = "gpt-5-nano"
REASONING = "high"
MLFLOW_EXPERIMENT_NAME = "cord-v2-gpt5-baseline"
PROMPT_NAME = f"invoice-extraction-prompt"
PROMPT_VERSION = "1"

### Initialize MLflow and OpenAI environment

In [3]:
client = OpenAI()
os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT_NAME
mlflow.openai.autolog()

### Register prompt and model

In [4]:
register_prompt(prompt_name=PROMPT_NAME)

You are a Vision Language Model designed to extract structured data from invoice receipts.
    Task:
    Convert the invoice receipt into a well-formed JSON object strictly following the schema provided.

    Requirements:
    1. Identify and extract only these sections (if present): `menu`, `sub_menu`, `sub_total`, `total`.  
    2. Preserve exact formatting for all the extracted values.  
    3. Do not output fields that lack data—omit empty keys.  
    4. Do not add any information not present in the invoice.
    5. In case of prices and currencies, ensure to maintain the original format without any modifications.

    Schema:
    {schema}

    Output:
    Return valid, minimal JSON matching this schema - no extraneous keys or null values.
    


2025/08/16 17:41:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: invoice-extraction-prompt, version 3


In [5]:
model_uri = log_invoice_extraction_model(model_name=MODEL_NAME, reasoning=REASONING, prompt_name=PROMPT_NAME, prompt_version=PROMPT_VERSION)

🏃 View run gpt-5-nano-high at: http://localhost:8080/#/experiments/417204914699444791/runs/ab0a8903186a4041a00b0fad98e36cc3
🧪 View experiment at: http://localhost:8080/#/experiments/417204914699444791


### Load the dataset

In [6]:
dataset = load_dataset("naver-clova-ix/cord-v2")
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 800
    })
    validation: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

### Data Preparation

In [7]:
with open("schema.json", "r") as f:
    schema_dict = json.load(f)

schema_dict

{'menu': {'nm': 'name of the menu',
  'num': 'identification number of menu',
  'unitprice': 'unit price of menu',
  'cnt': 'quantity of menu',
  'discountprice': 'discounted price of menu',
  'price': 'total price of menu',
  'itemsubtotal': 'price of each menu after discount applied',
  'vatyn': 'whether the price includes tax or not',
  'etc': 'others',
  'sub': {'nm': 'name of submenu',
   'unitprice': 'unit price of submenu',
   'cnt': 'quantity of submenu',
   'price': 'total price of submenu',
   'etc': 'others'}},
 'sub_total': {'price': 'subtotal price',
  'discount_price': 'discounted price in total',
  'service_price': 'service charge',
  'othersvc_price': 'added charge other than service charge',
  'tax_price': 'tax amount',
  'etc': 'others'},
 'total': {'total_price': 'total price',
  'etc': 'others',
  'cashprice': 'amount of price paid in cash',
  'changeprice': 'amount of change in cash',
  'creditcardprice': 'amount of price paid in credit/debit card',
  'emoneyprice'

In [8]:
test_dataset = dataset["test"]

url_list, ground_truth_list = generate_urls(dataset=test_dataset)

schema_dict_list = [schema_dict] * len(url_list)

eval_df = pd.DataFrame(
    {
        "schema": schema_dict_list,
        "image_base64": url_list,
    }
)

eval_df


100%|██████████| 100/100 [00:03<00:00, 27.03it/s]


,schema,image_base64
0,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
1,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
2,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
3,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
...,...,...
95,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
96,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
97,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
98,"{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


### Inference and Evaluation

In [9]:
if not os.path.exists("artifacts"):
    os.makedirs("artifacts")
    
eval_with_gt_df = pd.concat([pd.DataFrame({"ground_truth": ground_truth_list}), eval_df], axis=1)
eval_with_gt_df

,ground_truth,schema,image_base64
0,"{'menu': {'nm': '-TICKET CP', 'num': '901016',...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
1,"{'menu': [{'nm': 'J.STB PROMO', 'price': '1750...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
2,"{'menu': {'nm': 'JASMINE MT ( L )', 'cnt': '1'...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
3,"{'menu': {'nm': 'DONAT GULA', 'unitprice': '@1...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
4,"{'menu': [{'nm': 'ICE BLACKCOFFE', 'cnt': '2',...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
...,...,...,...
95,"{'menu': [{'nm': 'BASO TAHU', 'unitprice': '43...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
96,"{'menu': {'nm': 'BBQ Chicken', 'cnt': '1', 'pr...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
97,"{'menu': [{'nm': 'BUBUR GO', 'unitprice': '20....","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...
98,"{'menu': [{'nm': 'BLACK PAPPER MEATBALL PAS', ...","{'menu': {'nm': 'name of the menu', 'num': 'id...",/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcGBQgHBw...


In [10]:
def accuracy_metric(predictions, targets):
    predictions_list = [json.loads(pred) for pred in predictions]
    targets_list = targets.tolist()
    invoice_metrics_df = calculate_invoice_accuracies(targets_list, predictions_list)
    print(invoice_metrics_df)
    key_level_metrics_df = key_level_metrics(targets_list, predictions_list)
    accuracy_value = invoice_metrics_df["accuracy"].mean()
    print("Average accuracy:", accuracy_value)

    return accuracy_value

accuracy_metric = make_metric(
    eval_fn=accuracy_metric, greater_is_better=True, name="avg_accuracy"
)

In [11]:
results = mlflow.evaluate(
    model_uri,
    eval_with_gt_df,
    targets="ground_truth",  # specify which column corresponds to the expected output
    extra_metrics=[
        accuracy_metric,
    ],
)

2025/08/16 17:42:20 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-ee6244e96459479384d463c9460e42f2
2025/08/16 17:42:20 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/08/16 17:42:23 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-ee6244e96459479384d463c9460e42f2
2025/08/16 17:42:23 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2025/08/16 17:42:23 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/08/16 17:42:23 INFO mlflow.models.evaluation.evaluators.default: Computing model predictions.
2025/08/16 17:46:13 INFO mlflow.models.evaluation.default_evaluator: Testing metrics on first row...


   invoice_no  total_keys  matched_keys  accuracy
0           0          11             7  0.636364
False negative for key 'sub_total.subtotal_price': GT='60.000', Pred=None
False positive for key 'total.menutype_cnt': GT=None, Pred='1'
False positive for key 'menu.itemsubtotal': GT='60.000', Pred='60,000'
False positive for key 'menu.unitprice': GT=None, Pred='60.000'
False positive for key 'menu.nm': GT='-TICKET CP', Pred='TICKET CP'
False positive for key 'sub_total.price': GT=None, Pred='60.000'
False positive for key 'total.menuqty_cnt': GT='2.00', Pred='2'
Average accuracy: 0.6363636363636364
    invoice_no  total_keys  matched_keys  accuracy
0            0          11             7  0.636364
1            1           8             2  0.250000
2            2          10             3  0.300000
3            3           8             5  0.625000
4            4          14             2  0.142857
..         ...         ...           ...       ...
95          95          13           

[Trace(trace_id=tr-1daefa2edb18521dd0f6907e4e408aa8), Trace(trace_id=tr-514f437096ed1e5b1eb2a7adb16d5666), Trace(trace_id=tr-33a6c0a30d3492c18cdc021f4315724a), Trace(trace_id=tr-d18b55b946beca7161f39841644b08e7), Trace(trace_id=tr-67280d1243178eb8ffe1290c81b121a1), Trace(trace_id=tr-b2d736bca44f4cb66c952b5b69485a91), Trace(trace_id=tr-1cf379ccf5bd54342ba2e867a2bdbd4e), Trace(trace_id=tr-4c8d446915737d32289991182914558c), Trace(trace_id=tr-1a14fbf3e809144e75bb472205f0b413), Trace(trace_id=tr-2157aa7a536969657b42a921e05f3381)]

### Count number of tokens

In [12]:
trace_df  = mlflow.search_traces(run_id=results.run_id)
print(trace_df.shape)
total_input_tokens = 0
total_output_tokens = 0
total_reasoning_tokens = 0
for i in range(len(trace_df)):
    current_usage = trace_df["response"][i]['usage']
    total_input_tokens += current_usage['prompt_tokens']
    total_output_tokens += current_usage['completion_tokens']
    total_reasoning_tokens += current_usage['completion_tokens_details']['reasoning_tokens']

print("Total input tokens:", total_input_tokens)
print("Total output tokens:", total_output_tokens)
print("Total reasoning tokens:", total_reasoning_tokens)

(100, 12)
Total input tokens: 196198
Total output tokens: 336840
Total reasoning tokens: 322944


### Publish summary

In [13]:
with mlflow.start_run(run_name=f"{MODEL_NAME}-{REASONING}-summary") as run:
    mlflow.log_params({
        "model_name": MODEL_NAME,
    })

    # Log invoice_metrics_df
    mlflow.log_artifact("artifacts/invoice_metrics.csv")

    mlflow.log_artifact("artifacts/key_metrics.csv")

    # log the token usage
    mlflow.log_metric("input_tokens", total_input_tokens)
    mlflow.log_metric("output_tokens", total_output_tokens)
    

🏃 View run gpt-5-nano-high-summary at: http://localhost:8080/#/experiments/417204914699444791/runs/c35dcd8dd32d4bd08cbb79fc36f4965d
🧪 View experiment at: http://localhost:8080/#/experiments/417204914699444791
